# 03. Huấn luyện ba mô hình dự báo lượng mưa

Ba mô hình:

1. Linear Regression - baseline.
2. XGBoost - mô hình có khả năng học quan hệ phi tuyến.
3. GRU - mô hình chuỗi thời gian nâng cao.

Đầu vào và nhãn:

\[
X_{t-L+1:t}\in\mathbb{R}^{L\times d}
\longrightarrow
\hat y_{t+1}\in\mathbb{R}.
\]

Linear Regression và XGBoost dùng vector đặc trưng ở bước cuối của cửa sổ.
GRU dùng toàn bộ cửa sổ 144 bước, tương ứng 24 giờ với tần suất 10 phút.

## 1. Thư viện và cấu hình

In [ ]:
import json
import math
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

CONFIG = {
    "processed_dir": "../data/processed",
    "results_dir": "../results",
    "seq_len": 144,
    "horizon": 1,
    "rain_threshold_mm": 0.1,
    "prediction_threshold_mm": 0.1,
    "seed": 42,
    "xgb_n_estimators": 500,
    "xgb_learning_rate": 0.05,
    "xgb_max_depth": 6,
    "gru_hidden_size": 128,
    "gru_num_layers": 2,
    "gru_dropout": 0.25,
    "gru_head_size": 64,
    "gru_batch_size": 64,
    "gru_epochs": 60,
    "gru_patience": 8,
    "gru_learning_rate": 5e-4,
    "gru_weight_decay": 1e-5,
    "gru_huber_beta": 0.20,
    "max_rain_weight": 3.0,
}

PROCESSED_DIR = Path(CONFIG["processed_dir"])
RESULTS_DIR = Path(CONFIG["results_dir"])
MODELS_DIR = RESULTS_DIR / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", DEVICE)

## 2. Đọc ba tập dữ liệu đã xử lý

In [ ]:
required_files = [
    PROCESSED_DIR / "final_train.csv",
    PROCESSED_DIR / "final_val.csv",
    PROCESSED_DIR / "final_test.csv",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Hãy chạy 02_feature_engineering.ipynb trước. Thiếu: "
        + ", ".join(missing_files)
    )

train_df = pd.read_csv(required_files[0], parse_dates=["date"])
val_df = pd.read_csv(required_files[1], parse_dates=["date"])
test_df = pd.read_csv(required_files[2], parse_dates=["date"])

TARGET_COL = "rain_target_mm"
FEATURE_COLS = [
    col for col in train_df.columns
    if col not in ["date", TARGET_COL]
]

clean_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True,
)
x_all = clean_df[FEATURE_COLS].to_numpy(dtype=np.float32)
y_all = clean_df[TARGET_COL].to_numpy(dtype=np.float32)
timestamps = clean_df["date"].to_numpy()

train_end = len(train_df)
val_end = train_end + len(val_df)

print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)
print("Số đặc trưng:", len(FEATURE_COLS))

## 3. Tạo cửa sổ dự báo chung

In [ ]:
def build_split(target_start, target_end):
    first_target = max(CONFIG["seq_len"], target_start)
    target_positions = np.arange(
        first_target,
        target_end,
        dtype=np.int64,
    )
    starts = target_positions - CONFIG["horizon"] + 1
    targets = y_all[target_positions].astype(np.float32)
    tabular_inputs = x_all[target_positions - CONFIG["horizon"]]
    return starts, tabular_inputs, targets, target_positions

starts_train, Xtab_train, y_train, pos_train = build_split(
    0, train_end
)
starts_val, Xtab_val, y_val, pos_val = build_split(
    train_end, val_end
)
starts_test, Xtab_test, y_test, pos_test = build_split(
    val_end, len(clean_df)
)

print("Linear/XGBoost:", Xtab_train.shape, Xtab_val.shape, Xtab_test.shape)
print("Số cửa sổ GRU:", len(starts_train), len(starts_val), len(starts_test))

## 4. Linear Regression

In [ ]:
linear_model = LinearRegression()
linear_model.fit(Xtab_train, y_train)

pred_linear_raw = np.clip(
    linear_model.predict(Xtab_test),
    0,
    None,
)
joblib.dump(
    linear_model,
    MODELS_DIR / "linear_regression_rain.joblib",
)
print("Đã huấn luyện Linear Regression.")

## 5. XGBoost

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=CONFIG["xgb_n_estimators"],
    learning_rate=CONFIG["xgb_learning_rate"],
    max_depth=CONFIG["xgb_max_depth"],
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=CONFIG["seed"],
    n_jobs=-1,
)
xgb_model.fit(
    Xtab_train,
    y_train,
    eval_set=[(Xtab_val, y_val)],
    verbose=False,
)

pred_xgb_raw = np.clip(
    xgb_model.predict(Xtab_test),
    0,
    None,
)
joblib.dump(xgb_model, MODELS_DIR / "xgboost_rain.joblib")
print("Đã huấn luyện XGBoost.")

## 6. GRU một đầu ra

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, values, starts, targets):
        self.values = values
        self.starts = starts
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        start = int(self.starts[index])
        window = self.values[
            start - CONFIG["seq_len"]:start
        ]
        y_mm = max(float(self.targets[index]), 0.0)
        return (
            torch.from_numpy(window),
            torch.tensor([np.log1p(y_mm)], dtype=torch.float32),
            torch.tensor([y_mm], dtype=torch.float32),
        )

train_dataset = WindowDataset(x_all, starts_train, y_train)
val_dataset = WindowDataset(x_all, starts_val, y_val)
test_dataset = WindowDataset(x_all, starts_test, y_test)

loader_options = {
    "batch_size": CONFIG["gru_batch_size"],
    "num_workers": 0,
    "pin_memory": DEVICE.type == "cuda",
}
train_loader = DataLoader(
    train_dataset, shuffle=True, **loader_options
)
val_loader = DataLoader(
    val_dataset, shuffle=False, **loader_options
)
test_loader = DataLoader(
    test_dataset, shuffle=False, **loader_options
)

class GRUSingleOutput(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=CONFIG["gru_hidden_size"],
            num_layers=CONFIG["gru_num_layers"],
            batch_first=True,
            dropout=(
                CONFIG["gru_dropout"]
                if CONFIG["gru_num_layers"] > 1 else 0.0
            ),
        )
        self.norm = nn.LayerNorm(CONFIG["gru_hidden_size"])
        self.head = nn.Sequential(
            nn.Linear(
                CONFIG["gru_hidden_size"],
                CONFIG["gru_head_size"],
            ),
            nn.GELU(),
            nn.Dropout(CONFIG["gru_dropout"]),
            nn.Linear(CONFIG["gru_head_size"], 1),
        )

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.head(self.norm(hidden[-1]))

gru_model = GRUSingleOutput(len(FEATURE_COLS)).to(DEVICE)
print(
    "Số tham số GRU:",
    sum(
        parameter.numel()
        for parameter in gru_model.parameters()
        if parameter.requires_grad
    ),
)

In [ ]:
train_rain = y_train
n_rain = int(np.sum(train_rain >= CONFIG["rain_threshold_mm"]))
n_dry = len(train_rain) - n_rain
if n_rain == 0:
    raise ValueError("Tập train không có mẫu mưa.")

RAIN_WEIGHT = min(
    math.sqrt(n_dry / n_rain),
    CONFIG["max_rain_weight"],
)
RAIN_WEIGHT = max(RAIN_WEIGHT, 1.0)

huber = nn.SmoothL1Loss(
    beta=CONFIG["gru_huber_beta"],
    reduction="none",
)
optimizer = torch.optim.AdamW(
    gru_model.parameters(),
    lr=CONFIG["gru_learning_rate"],
    weight_decay=CONFIG["gru_weight_decay"],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

def weighted_gru_loss(pred_log, target_log, target_mm):
    losses = huber(pred_log, target_log)
    weights = torch.where(
        target_mm >= CONFIG["rain_threshold_mm"],
        torch.full_like(target_mm, RAIN_WEIGHT),
        torch.ones_like(target_mm),
    )
    return (losses * weights).sum() / weights.sum()

def run_gru_epoch(loader, training):
    gru_model.train(training)
    total_loss = 0.0
    total_n = 0

    for x, y_log, y_mm in loader:
        x = x.to(DEVICE, non_blocking=True)
        y_log = y_log.to(DEVICE, non_blocking=True)
        y_mm = y_mm.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        pred_log = gru_model(x)
        loss = weighted_gru_loss(pred_log, y_log, y_mm)

        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                gru_model.parameters(),
                1.0,
            )
            optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)

    return total_loss / max(total_n, 1)

history = {"epoch": [], "train_loss": [], "val_loss": []}
best_val = float("inf")
bad_epochs = 0
best_model_path = MODELS_DIR / "best_gru_rain.pt"

for epoch in range(1, CONFIG["gru_epochs"] + 1):
    train_loss = run_gru_epoch(train_loader, True)
    with torch.no_grad():
        val_loss = run_gru_epoch(val_loader, False)

    scheduler.step(val_loss)
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val - 1e-8:
        best_val = val_loss
        bad_epochs = 0
        torch.save(gru_model.state_dict(), best_model_path)
        status = "saved"
    else:
        bad_epochs += 1
        status = f"patience {bad_epochs}/{CONFIG['gru_patience']}"

    print(
        f"Epoch {epoch:02d} | train {train_loss:.6f} | "
        f"val {val_loss:.6f} | {status}"
    )

    if bad_epochs >= CONFIG["gru_patience"]:
        print("Early stopping")
        break

pd.DataFrame(history).to_csv(
    RESULTS_DIR / "gru_training_history.csv",
    index=False,
)

gru_model.load_state_dict(
    torch.load(best_model_path, map_location=DEVICE)
)

@torch.no_grad()
def predict_gru(loader):
    gru_model.eval()
    predictions = []
    for x, _, _ in loader:
        pred_log = (
            gru_model(x.to(DEVICE))
            .cpu()
            .numpy()
            .reshape(-1)
        )
        predictions.append(
            np.clip(np.expm1(pred_log), 0, None)
        )
    return np.concatenate(predictions)

pred_gru_raw = predict_gru(test_loader)
print("Đã huấn luyện GRU.")

## 7. Lưu dự báo của ba mô hình

In [ ]:
def apply_threshold(values):
    result = np.asarray(values, dtype=float).copy()
    result[
        result < CONFIG["prediction_threshold_mm"]
    ] = 0.0
    return result

pred_linear = apply_threshold(pred_linear_raw)
pred_xgb = apply_threshold(pred_xgb_raw)
pred_gru = apply_threshold(pred_gru_raw)

assert len(pred_linear) == len(y_test)
assert len(pred_xgb) == len(y_test)
assert len(pred_gru) == len(y_test)

predictions_df = pd.DataFrame({
    "target_time": pd.to_datetime(timestamps[pos_test]),
    "y_true_mm": y_test,
    "linear_regression_mm": pred_linear,
    "xgboost_mm": pred_xgb,
    "gru_mm": pred_gru,
})
predictions_df.to_csv(
    RESULTS_DIR / "three_models_predictions.csv",
    index=False,
)

with (RESULTS_DIR / "model_config.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(CONFIG, handle, ensure_ascii=False, indent=2)

print("Đã lưu mô hình và dự báo vào:", RESULTS_DIR)
display(predictions_df.head())